# Dengue en Loreto (2000–2024)
## Notebook 05 · Exportación para Power BI

**Entrada:** `data/dengue_loreto.db`, generada en `03_sql.ipynb`.

**Qué hace este notebook:** exporta las tablas de la base de datos a archivos CSV para construir el dashboard en Power BI, siguiendo un **modelo estrella**:

- `casos`: **tabla de hechos**, un caso de dengue por fila.
- `poblacion`: **dimensión de distritos**, con la población de cada uno.
- `poblacion_edad`: **dimensión de grupos de edad**, con su población y una columna `orden` para ordenarlos en los gráficos.

**Resultado:** archivos CSV en `data/powerbi/` (no se suben a GitHub).

In [1]:
import os
import sqlite3
import pandas as pd

conexion = sqlite3.connect("../data/dengue_loreto.db")
os.makedirs("../data/powerbi", exist_ok=True)

casos = pd.read_sql("SELECT * FROM casos", conexion)
poblacion = pd.read_sql("SELECT * FROM poblacion", conexion)
poblacion_edad = pd.read_sql("SELECT * FROM poblacion_edad", conexion)

# Orden de los grupos de edad (Power BI ordena el texto alfabéticamente)
orden = {
    "Niño (0-11)": 1,
    "Adolescente (12-17)": 2,
    "Joven (18-29)": 3,
    "Adulto (30-59)": 4,
    "Adulto mayor (60+)": 5,
}
poblacion_edad["orden"] = poblacion_edad["grupo_edad"].map(orden)

tablas = {"casos": casos, "poblacion": poblacion, "poblacion_edad": poblacion_edad}

for nombre, df in tablas.items():
    df.to_csv(f"../data/powerbi/{nombre}.csv", index=False, encoding="utf-8-sig")
    print(f"{nombre}: {len(df):,} filas")

casos: 107,074 filas
poblacion: 53 filas
poblacion_edad: 5 filas


### Validación

Se comprueba que cada CSV tenga las mismas filas que su tabla en la base de datos y que los totales de casos y población no hayan cambiado.

In [2]:
# Releer los CSV exportados y comparar con la base de datos
for nombre, df in tablas.items():
    exportado = pd.read_csv(f"../data/powerbi/{nombre}.csv", encoding="utf-8-sig")
    assert len(exportado) == len(df), f"{nombre}: el número de filas no coincide"

assert len(casos) == 107074
assert poblacion["poblacion"].sum() == poblacion_edad["poblacion"].sum() == 1167197

print("✅ Exportación validada")

✅ Exportación validada


---
## Modelo de datos en Power BI

Modelo estrella con la tabla de hechos `casos` en el centro y tres dimensiones. Todas las relaciones son de varios a uno (*:1), con filtro en una sola dirección: de la dimensión hacia los casos.

| Dimensión (lado 1) | Hechos (lado *) | Para qué sirve |
|---|---|---|
| `poblacion[ubigeo]` | `casos[ubigeo]` | Incidencia por distrito y provincia |
| `poblacion_edad[grupo_edad]` | `casos[grupo_edad]` | Incidencia por grupo de edad |
| `Anios[ano]` | `casos[ano]` | Filtro de años y número de años seleccionados |

`Anios` es una tabla calculada en Power BI: `Anios = DISTINCT(casos[ano])`. En la vista de tabla, `grupo_edad` se ordena por la columna `orden`.

**Tipos de datos:** `ubigeo` se carga como texto en `casos` y en `poblacion`, porque una relación solo funciona entre columnas del mismo tipo.

## Medidas DAX

```dax
Casos = COUNTROWS(casos)
Población = SUM(poblacion[poblacion])
Población por edad = SUM(poblacion_edad[poblacion])
N años = COUNTROWS(VALUES(Anios[ano]))
Incidencia = DIVIDE([Casos] * 100000, [Población] * [N años])
Incidencia por edad = DIVIDE([Casos] * 100000, [Población por edad] * [N años])
% con alarma o grave = CALCULATE(AVERAGE(casos[con_alarma_o_grave]), KEEPFILTERS(Anios[ano] >= 2010))
```

**Decisiones:**

- **`N años` cuenta los años seleccionados en el filtro**, no los años con casos (`DISTINCTCOUNT(casos[ano])`). Si no, en un distrito con casos solo en 2 de 5 años, la incidencia se dividiría entre 2 y saldría inflada.
- **Dos medidas de población:** la de distritos para la incidencia geográfica y la de grupos de edad para la incidencia por edad.
- **`KEEPFILTERS`** limita el porcentaje de gravedad a 2010–2024 sin anular el filtro de años elegido por el usuario.

## Validación contra SQL

Con el filtro en 2020–2024, las medidas de Power BI coinciden con las consultas de `03_sql.ipynb` (sección 10.4):

| | SQL | Power BI |
|---|---|---|
| Casos totales | 36,419 | 36,419 |
| Incidencia de Loreto | 624.0 | 624.0 |
| Yurimaguas (casos / incidencia) | 11,941 / 2,456.4 | 11,941 / 2,456.4 |

Con el filtro en 2000–2024, aparece una fila sin distrito: son casos con códigos antiguos que no existen en la tabla de población de 2021. Por eso la incidencia se reporta solo para 2020–2024.